# 00 · OpenEnv Quickstart — connect, step, read reward (CPU-only)

**What is OpenEnv?** A standard for *execution environments* an LLM agent acts in.
Each environment exposes a Gymnasium-style surface — `reset()` to start an episode,
`step(action)` to act, and a `state` — served over **HTTP** and packaged so it can run
as a **Hugging Face Space**. The agent never holds the env in-process; it talks to a
server. That decoupling is what lets the same env drive training (TRL) and evaluation.

This first notebook uses **no GPU and trains nothing**. You will connect to a hosted
environment, drive it by hand with a few `step()` calls, and read the reward it returns.
That is the whole contract the trainer automates later (notebooks 01 and 02).

> **Runtime targets — this notebook runs anywhere.** Colab (A100/L4/T4), a fresh
> local GPU box, or an HPC cluster via the SLURM runner in `jobs/`. It self-installs
> its dependencies, authenticates with `notebook_login()` (or `HF_TOKEN`), connects
> to an environment by **URL** (a variable you can repoint to your own Space), and
> auto-detects the GPU with a model-size fallback. There are **no hard-coded paths**.

<!-- @inject:run-recipes -->
## ▶ How to run this notebook off-DRAC (HF Jobs · Google Colab)

This notebook is portable — **no DRAC required**. Pick a lane. (Dependencies and the env
URL below are taken from this notebook's own cells.)

### A · HF Jobs — run the notebook on Hugging Face infra

`hf jobs run` is *docker-run for HF infra*: the compute happens on HF's cloud, you only
submit. The control plane is plain HTTPS, so this works even from networks that block the
hosted-env WebSocket (e.g. DRAC compute). Requires pre-paid credits on your HF account.

```bash
# one-time: hf auth login   (or rely on a local HF_TOKEN)
hf jobs run \
  --flavor cpu-basic \
  --timeout 1800 \
  \
  -e ENV_BASE_URL=https://sergiopaniego-reasoning-gym.hf.space \
  python:3.12 \
  bash -c "pip install -q papermill && \
           pip install -q openenv "transformers>=5.3.0" && \
           pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym && \
           papermill 00_openenv_quickstart.ipynb out.ipynb"
```

Notes: `--secrets HF_TOKEN` forwards your local token (needed if the notebook pushes to the
Hub). `--flavor` options: `cpu-basic`, `t4-small`, `a10g-small`, `a100-large`, … (`hf jobs`
docs). Add `--detach` to background it; watch with `hf jobs logs <id>`.

### C · Google Colab — independent run

Open Colab → set runtime to **CPU (no GPU needed)** → in the first cell:

```python
# Colab cell 1 — auth + knobs, then run the rest of the notebook top-to-bottom.
import os
os.environ["SMOKE"] = "0"                 # full run (omit for a quick smoke)
os.environ["ENV_BASE_URL"] = "https://sergiopaniego-reasoning-gym.hf.space"
from huggingface_hub import notebook_login
notebook_login()                          # paste a token with write access
```

Then **Runtime → Run all**. Colab has open egress, so the hosted env Space works directly
(no in-job uvicorn needed). The install cell at the top of the notebook handles
dependencies.

---


## 1 · Install

Just the client libraries and the environment package. The env package is a thin
client pulled directly from its Space; `--no-deps` keeps it from dragging in a heavy
server-side stack you do not need on the client.

In [ ]:
# reasoning_gym is our example env; its client ships from the Space.
%pip install -q openenv "transformers>=5.3.0"
%pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym

In [ ]:
# notebooks drive async env clients through a sync wrapper; nest_asyncio lets the
# event loop run inside IPython.
import nest_asyncio

nest_asyncio.apply()

## 2 · Connect to a hosted environment

We point the client at a **URL**. The default is the tutorial's hosted Space; set
`ENV_BASE_URL` in your environment to use your own duplicated Space instead. The
`.sync()` wrapper turns the async client into blocking calls so we can step it inline.

In [ ]:
import os

# Repoint this to your own Space (duplicate the hosted one) for real workloads —
# hosted tutorial Spaces are low-concurrency.
ENV_BASE_URL = os.environ.get("ENV_BASE_URL", "https://sergiopaniego-reasoning-gym.hf.space")
print(f"Connecting to: {ENV_BASE_URL}")

from reasoning_gym_env import ReasoningGymAction, ReasoningGymEnv

env = ReasoningGymEnv(base_url=ENV_BASE_URL).sync()

## 3 · `reset()` — start an episode

`reset()` configures the procedural task (here `chain_sum`: a chain of integer
additions) and returns the first observation. *Procedural* means the env generates a
fresh problem each episode, so a trained model must **generalize** over the family
rather than memorize fixed items.

In [ ]:
DATASET_CONFIG = {"min_terms": 2, "max_terms": 3, "min_digits": 2, "max_digits": 2}

result = env.reset(
    dataset_name="chain_sum",
    dataset_config=DATASET_CONFIG,
    seed=0,
    size=10,
)
print("Question:", result.observation.question)

## 4 · `step(action)` — act and read the reward

The env exposes an `answer` tool. We submit an answer as a `ReasoningGymAction` and
the env returns the score and the correct answer. This scalar reward is the *only*
signal GRPO needs in notebook 01.

In [ ]:
# Solve it correctly to see reward = 1.0. (We cheat here by evaluating the printed
# arithmetic; a real agent would reason it out.)
import re

nums = [int(n) for n in re.findall(r"-?\d+", result.observation.question)]
my_answer = str(sum(nums)) if nums else "0"

step = env.step(ReasoningGymAction(answer=my_answer))
print(f"submitted={my_answer}")
print(f"score={step.observation.score}  correct={step.observation.correct_answer}")

## 5 · A few episodes by hand

After the first configured `reset()`, calling `reset()` with no args advances to the
next question in the same dataset iterator. This loop is exactly what the trainer runs
per rollout — just automated and batched.

In [ ]:
score_sum, n = 0.0, 5
for i in range(n):
    r = env.reset() if i else result  # reuse the first question we already pulled
    q = r.observation.question
    nums = [int(x) for x in re.findall(r"-?\d+", q)]
    ans = str(sum(nums)) if nums else "0"
    s = env.step(ReasoningGymAction(answer=ans))
    print(f"[{i}] {q!r:55s} -> ans={ans:>5s}  score={s.observation.score}")
    score_sum += float(s.observation.score or 0.0)

print(f"\nmean score over {n} episodes: {score_sum / n:.2f}")

## What you just learned

- An OpenEnv environment is an **HTTP server** with a `reset` / `step` contract; the
  client is a thin wrapper you connect to by URL.
- The env returns a **scalar reward** per step. That is the entire interface RL needs.
- Everything here was **manual**. In notebook 01, TRL's `GRPOTrainer` does this loop
  for you via the `environment_factory` pattern — creating an env per rollout, parsing
  the model's tool call, stepping, and collecting the reward.

**Next:** `01_reasoning_gym_sft_then_grpo.ipynb` — collect teacher rollouts, SFT
warm-start, then GRPO, then test the trained agent against held-out problems.